# BC Policy Evaluation — NDCG Improvement vs Cost

This notebook loads the trained `bc_policy.pt` (behavioral cloning policy) and runs it in the
retrieval simulation environment on 30 unseen MS MARCO dev queries, mirroring the approach in
`evaluate_cql_unseen_queries.ipynb`.

It records per-step NDCG@50 and action cost, then plots **cumulative cost** (x-axis) vs
**NDCG improvement percentage** (y-axis) per query.

**Requirements**: `.env` with `RETRIEVAL_ENDPOINT`, `MY_USERNAME`, `MY_PASSWORD`; `bc_policy.pt`;
`data/topics.ms-marco-dev.tsv`; `data/qrels.ms-marco-dev.tsv`.

In [7]:
from __future__ import annotations
import sys
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from sentence_transformers import SentenceTransformer
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.agents.prf import PRFAgent
from src.agents.reformulate import ReformulationAgent
from src.agents.rerank import RerankingAgent
from src.simulation import ACTION_NAMES, ACTION_STOP, SimConfig, Simulation
from src.utils.retriever import Retriever, create_retriever_callable
TOPICS_PATH = ROOT / 'data/topics.ms-marco-dev.tsv'
QRELS_PATH = ROOT / 'data/qrels.ms-marco-dev.tsv'
MODEL_PATH = ROOT / 'bc_policy.pt'
RESULTS_PATH = ROOT / 'eval_bc_30_unseen_queries.csv'
MAX_STEPS, TOP_K, NDCG_K = 3, 50, 50
INPUT_DIM, HIDDEN_DIMS, NUM_ACTIONS = 399, (256, 128), 4

In [8]:
topics = pd.read_csv(TOPICS_PATH, sep='\t', dtype={'query_id': str}).drop_duplicates('query_id').head(30).reset_index(drop=True)
judgments = pd.read_csv(QRELS_PATH, sep='\t', dtype={'query_id': str, 'doc_id': str})
qrels = defaultdict(dict)
for row in judgments.itertuples(index=False): qrels[str(row.query_id)][str(row.doc_id)] = int(row.relevance)
qrels = dict(qrels)
if len(topics) != 30: raise ValueError(f'Expected 30 topics, found {len(topics)}')
topics = topics[topics.query_id.isin(qrels)].reset_index(drop=True)
if len(topics) != 30: raise ValueError('Some selected topics have no qrels')
print(f'{len(topics)} queries with qrels ready')

30 queries with qrels ready


In [9]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
encoder = SentenceTransformer('all-MiniLM-L6-v2')
retriever = create_retriever_callable(Retriever())
agents = [ReformulationAgent(encoder), RerankingAgent(encoder), PRFAgent(encoder, num_expansion_terms=5)]
sim = Simulation(encoder, retriever, agents, SimConfig(max_steps=MAX_STEPS, top_k_rerank=TOP_K, ndcg_k=NDCG_K))
_probe_ids, _, _ = retriever('test query', 1)
if not _probe_ids:
    print('WARNING: retriever returned no documents for a test query.')
    print('Check that RETRIEVAL_ENDPOINT in .env is reachable and credentials are correct.')
print(f'Simulation ready; device={device}')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2082.19it/s]
Error retrieving documents: HTTPSConnectionPool(host='opensearch.pads.fim.uni-passau.de', port=443): Max retries exceeded with url: /msmarco-v2.1-segmented/_search (Caused by NameResolutionError("HTTPSConnection(host='opensearch.pads.fim.uni-passau.de', port=443): Failed to resolve 'opensearch.pads.fim.uni-passau.de' ([Errno 11001] getaddrinfo failed)"))


Check that RETRIEVAL_ENDPOINT in .env is reachable and credentials are correct.
Simulation ready; device=cuda:0


In [10]:
class BCPolicy(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: tuple[int, ...], num_actions: int):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, num_actions))
        self.net = nn.Sequential(*layers)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

model = BCPolicy(INPUT_DIM, HIDDEN_DIMS, NUM_ACTIONS).to(device)
ckpt = torch.load(str(MODEL_PATH), map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f"Loaded BC policy from {MODEL_PATH} (epoch {ckpt.get('epoch', '?')}, best_test_acc={ckpt.get('best_test_accuracy', '?')})")

Loaded BC policy from c:\Users\hanaz\Documents\GitHub\Agentic-Retrieval\bc_policy.pt (epoch 7, best_test_acc=0.6151241660118103)


In [11]:
def evaluate(query_id, query_text):
    ids, scores, corpus = retriever(query_text, TOP_K)
    if not ids:
        raise RuntimeError(f"Retriever returned 0 documents for '{query_text[:40]}'. "
            "Verify RETRIEVAL_ENDPOINT in .env is reachable.")
    original = encoder.encode(query_text, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    query, previous, embedding = query_text, None, original.copy(); last = [False, False, False]; total_cost = total_latency = 0.0; rows = []
    for step in range(MAX_STEPS):
        state = sim.build_state(query, ids, scores, step, last, previous, original, total_latency, total_cost, len(query.split()), embedding)
        before = sim.compute_ndcg(ids, qrels[query_id], NDCG_K); valid = sim._valid_actions_mask(last)
        with torch.no_grad():
            raw = int(model(torch.from_numpy(state).float().to(device)).argmax().item())
        action = raw if valid[raw] else ACTION_STOP
        new_query, new_ids, new_scores, metrics, latency, cost = sim.compute_effects(action, query, ids, scores, qrels[query_id], corpus_data=corpus)
        cost = float(cost or 0.0); after = float(metrics['ndcg']); total_cost += cost; total_latency += float(latency)
        rows.append({'query_id': query_id, 'query_text': query_text, 'step': step, 'raw_action': raw, 'raw_action_name': ACTION_NAMES[raw], 'action': action, 'action_name': ACTION_NAMES[action], 'mask_override': raw != action, 'ndcg_before': before, 'ndcg_after': after, 'ndcg_gain': after-before, 'cost': cost, 'cumulative_cost': total_cost, 'latency_ms': float(latency)})
        if action == ACTION_STOP: break
        previous, query, ids, scores = list(ids), new_query, list(new_ids), np.asarray(new_scores, dtype=np.float32)
        embedding = encoder.encode(query, convert_to_numpy=True, show_progress_bar=False).astype(np.float32); last = [action == 0, action == 1, action == 2]
    return rows

In [12]:
records, errors = [], []
for row in topics.itertuples(index=False):
    try: records.extend(evaluate(str(row.query_id), row.query_text))
    except Exception as exc: errors.append({'query_id': str(row.query_id), 'error': repr(exc)})
print(f'queries={len(topics)}, failures={len(errors)}, total_steps={len(records)}')
if errors: print(errors[:3])

Error retrieving documents: HTTPSConnectionPool(host='opensearch.pads.fim.uni-passau.de', port=443): Max retries exceeded with url: /msmarco-v2.1-segmented/_search (Caused by NameResolutionError("HTTPSConnection(host='opensearch.pads.fim.uni-passau.de', port=443): Failed to resolve 'opensearch.pads.fim.uni-passau.de' ([Errno 11001] getaddrinfo failed)"))
Error retrieving documents: HTTPSConnectionPool(host='opensearch.pads.fim.uni-passau.de', port=443): Max retries exceeded with url: /msmarco-v2.1-segmented/_search (Caused by NameResolutionError("HTTPSConnection(host='opensearch.pads.fim.uni-passau.de', port=443): Failed to resolve 'opensearch.pads.fim.uni-passau.de' ([Errno 11001] getaddrinfo failed)"))
Error retrieving documents: HTTPSConnectionPool(host='opensearch.pads.fim.uni-passau.de', port=443): Max retries exceeded with url: /msmarco-v2.1-segmented/_search (Caused by NameResolutionError("HTTPSConnection(host='opensearch.pads.fim.uni-passau.de', port=443): Failed to resolve 'op

queries=30, failures=30, total_steps=0
[{'query_id': '2', 'error': 'RuntimeError("Retriever returned 0 documents for \'Androgen receptor define\'. Verify RETRIEVAL_ENDPOINT in .env is reachable.")'}, {'query_id': '524332', 'error': 'RuntimeError("Retriever returned 0 documents for \'treating tension headaches without medic\'. Verify RETRIEVAL_ENDPOINT in .env is reachable.")'}, {'query_id': '1048642', 'error': 'RuntimeError("Retriever returned 0 documents for \'what is paranoid sc\'. Verify RETRIEVAL_ENDPOINT in .env is reachable.")'}]


In [ ]:
results = pd.DataFrame(records)
if results.empty:
    print('All queries failed. Errors:')
    for e in errors:
        print(f"  q{e['query_id']}: {e['error'][:120]}")
    raise RuntimeError('No query was evaluated successfully. Ensure the OpenSearch endpoint in .env is reachable.')
results.to_csv(RESULTS_PATH, index=False)
print(results.action_name.value_counts())
print(f'Saved per-step records to {RESULTS_PATH}')

In [ ]:
initial_ndcg = results[results.step == 0].groupby('query_id')['ndcg_before'].first()
final_ndcg = results.groupby('query_id')['ndcg_after'].last()
cumulative_cost = results.groupby('query_id')['cumulative_cost'].max()
plot_df = pd.DataFrame({'initial_ndcg': initial_ndcg, 'final_ndcg': final_ndcg, 'cumulative_cost': cumulative_cost})
plot_df['ndcg_improvement_pct'] = (plot_df['final_ndcg'] - plot_df['initial_ndcg']) / plot_df['initial_ndcg'].clip(lower=1e-9) * 100
plot_df = plot_df.reset_index().rename(columns={'query_id': 'query_id'})
print(plot_df[['initial_ndcg', 'final_ndcg', 'ndcg_improvement_pct', 'cumulative_cost']].describe())
plot_df

## Cumulative Cost vs NDCG Improvement

Each point is one of the 30 unseen queries. The x-axis is total resource cost
(sum of `ACTION_COSTS` + reformulation token cost across the trajectory);
the y-axis is NDCG improvement from the BM25 baseline to the final state.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].axhline(y=0, color='gray', linestyle='--', linewidth=0.8, label='baseline (0%)')
axes[0].scatter(plot_df['cumulative_cost'], plot_df['ndcg_improvement_pct'], alpha=0.7, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel('Cumulative Cost')
axes[0].set_ylabel('NDCG Improvement (%)')
axes[0].set_title('BC Policy: NDCG Improvement vs Cumulative Cost (30 unseen queries)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(plot_df['ndcg_improvement_pct'], bins=15, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='gray', linestyle='--', linewidth=0.8, label='baseline (0%)')
axes[1].set_xlabel('NDCG Improvement (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of NDCG Improvement (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()